In [38]:
import json
import os
import re
import time
import urllib.request
import urllib.error
from collections import Counter

import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv(override=True)

print('라이브러리 로드 완료')

라이브러리 로드 완료


In [39]:
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY','').strip()
OPENAI_MODEL = 'gpt-5-nano'
HF_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

print(OPENAI_API_KEY[:10] + '...'+OPENAI_API_KEY[-10:])

sk-proj-IF...uEOf8jXrMA


In [40]:
# 평가 데이터

sample_data = [
    {'id': 'D01', 'category': 'DEF', 'text': '이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.'},
    {'id': 'D02', 'category': 'DEF', 'text': '이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.'},
    {'id': 'D03', 'category': 'DEF', 'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.'},
    {'id': 'D04', 'category': 'DEF', 'text': '이 법은 대한민국 영역 안에서 이루어지는 정보 처리 행위에 적용한다.'},
    {'id': 'R01', 'category': 'RIGHT', 'text': '모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.'},
    {'id': 'R02', 'category': 'RIGHT', 'text': '사업자는 이용자의 개인정보를 안전하게 관리하여야 한다.'},
    {'id': 'R03', 'category': 'RIGHT', 'text': '근로자는 안전하고 건강한 근무 환경에서 일할 권리를 가진다.'},
    {'id': 'R04', 'category': 'RIGHT', 'text': '누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.'},
    {'id': 'P01', 'category': 'PROC', 'text': '이 법을 위반한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처한다.'},
    {'id': 'P02', 'category': 'PROC', 'text': '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.'},
    {'id': 'P03', 'category': 'PROC', 'text': '장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.'},
    {'id': 'P04', 'category': 'PROC', 'text': '불법행위로 인한 손해배상 청구는 민사소송법에서 정한 절차에 따른다.'},
    {'id': 'O01', 'category': 'ORG', 'text': '분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.'},
    {'id': 'O02', 'category': 'ORG', 'text': '위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.'},
    {'id': 'O03', 'category': 'ORG', 'text': '법원은 사법권을 행사하며 대법원, 고등법원 및 지방법원으로 구성된다.'},
    {'id': 'O04', 'category': 'ORG', 'text': '중앙행정기관의 장은 소관 사무를 관장하고 소속 공무원을 지휘한다.'},
    {'id': 'C01', 'category': 'CRIT', 'text': '후보자는 선거일 현재 25세 이상인 국민이어야 한다.'},
    {'id': 'C02', 'category': 'CRIT', 'text': '지원 자격은 해당 분야 경력 3년 이상 및 학사 학위 이상으로 한다.'},
    {'id': 'C03', 'category': 'CRIT', 'text': '안전관리 기준은 시설 면적, 이용 인원 및 위험도에 따라 대통령령으로 정한다.'},
    {'id': 'C04', 'category': 'CRIT', 'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.'},
    {'id': 'E01', 'category': 'ETC', 'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.'},
    {'id': 'E02', 'category': 'ETC', 'text': '이 법 시행 당시 종전의 규정에 따라 한 처분은 이 법에 따른 처분으로 본다.'},
    {'id': 'E03', 'category': 'ETC', 'text': '이 법의 시행에 필요한 사항은 대통령령으로 정한다.'},
    {'id': 'E04', 'category': 'ETC', 'text': '제3조의 개정규정은 이 법 시행 후 최초로 접수된 사건부터 적용한다.'},
]

label_desc = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항'
}

MAX_SAMPLES = 24

df_all = pd.DataFrame(sample_data)
df = df_all.head(MAX_SAMPLES).copy()
print(f'평가 데이터: {len(df)}건 / 전체 {len(df_all)}건')
df.head(5)

평가 데이터: 24건 / 전체 24건


,id,category,text
0,D01,DEF,이 법은 국민의 기본적 인권을 보호하고 자유와 평등을 실현함을 목적으로 한다.
1,D02,DEF,이 법에서 사용하는 용어의 뜻은 다음 각 호와 같다.
2,D03,DEF,"공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다."
3,D04,DEF,이 법은 대한민국 영역 안에서 이루어지는 정보 처리 행위에 적용한다.
4,R01,RIGHT,"모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 ..."


In [41]:
# 프롬프트 템플릿 정의(zero shot / one shot)
ALLOWED = list(label_desc.keys())

SYSTEM_PROMPT = (
    '너는 한국 법률 조항 분류기다, 반드시 다음 6개 코드 중 하나만 선택하라' + 
    ', '.join(ALLOWED) + '. '
    '응답은 반드시 JSON 한 줄로만 반환한다. 형식: {"label" : "DEF","reason":"..."}'
)

LABEL_GUIDE = '\n'.join([f'-{k}:{v}'  for k, v in label_desc.items()])

ONE_SHOT_EXAMPLE = {
    'text' : '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다',
    'output' : {'label':'PROC', 'reason' : '이의신청과 처리 절차를 규정한다'}
}

def build_user_prompt(text, mode='zero'):
    base = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하라.',
        '라벨 설명:',
        LABEL_GUIDE,
    ]
    if mode == 'one':
        base += [
            '',
            '예시 1개:',
            f'문장: {ONE_SHOT_EXAMPLE["text"]}',
            '정답(JSON): ' + json.dumps(ONE_SHOT_EXAMPLE['output'], ensure_ascii=False)
        ]
    base += ['', f'분류할 문장: {text}', 'JSON으로만 응답하라.']
    return '\n'.join(base)

print(build_user_prompt('위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.', mode='zero')[:300])


다음 법률 문장을 6개 코드 중 하나로 분류하라.
라벨 설명:
-DEF:정의/목적/적용범위 조항
-RIGHT:권리/의무/금지/책임 조항
-PROC:신청/심사/조사/불복/처벌 절차 조항
-ORG:기관/위원회/법원 등 조직의 설치/구성/권한 조항
-CRIT:자격/요건/기준/기간/수치 조건 조항
-ETC:시행일/경과조치/위임 등 기타 조항

분류할 문장: 위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.
JSON으로만 응답하라.


In [42]:
# openai 호출
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

user_content = build_user_prompt('위원회는 위원장 1명을 포함한 15명 이내의 위원으로 구성한다.', mode='zero')
messages = [
    {'role':'system','content':SYSTEM_PROMPT},
    {'role':'user','content':user_content}
]

response = client.responses.create(
    model=OPENAI_MODEL,
    input=messages
)

print(response.output_text)

{"label" : "ORG","reason":"위원회의 설치 및 구성을 다루는 구성 인원 규정에 해당하는 조항"}


In [43]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
# messages = [
#     {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
#     {"role": "user", "content": prompt}
# ]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

{"label": "ORG", "reason": "합법적인 구성 요소가 있는 문장입니다."}


In [44]:
# 허깅페이스모델로..  few shot을 만들어서 실습

In [45]:
import json
from typing import Dict, List, Tuple

FEW_SHOT_EXAMPLES = [
    {
        'text': '신청인은 처분 통지를 받은 날부터 30일 이내에 이의신청을 할 수 있다.',
        'output': {'label': 'PROC', 'reason': '신청/불복 절차를 규정하는 조항'}
    },
    {
        'text': '근로자는 안전하고 건강한 근무 환경에서 일할 권리를 가진다.',
        'output': {'label': 'RIGHT', 'reason': '근로자의 권리를 규정하는 조항'}
    },
    {
        'text': '분쟁 조정을 위하여 국무총리 소속으로 조정위원회를 둔다.',
        'output': {'label': 'ORG', 'reason': '조직의 설치를 규정하는 조항'}
    }
]

In [46]:
def build_hf_prompt(text:str, mode:str='zero', examples:List[Dict]=None) -> str:
    base = [
        '다음 법률 문장을 6개 코드 중 하나로 분류하세요',
        '라벨 설명:',
        LABEL_GUIDE
    ]
    if mode == 'one':
        base += [
            "",
            "[One-Shot 예시 1개]",
            f'문장:{ONE_SHOT_EXAMPLE["text"]}',
            "정답(JSON): " + json.dumps(ONE_SHOT_EXAMPLE['output'], ensure_ascii=False)
        ]
    elif mode =='few' and examples:
        base += ["", f'[Few-Shot 예시 {len(examples)}]개']
        for i , ex in enumerate(examples, 1):
            base += [
                f'예시 {i}',
                f' 문장 : {ex["text"]}',
                f' 정답 :' + json.dumps(ex['output'], ensure_ascii=False)
            ]
    base += [
        "",
        f"분류할 문장: {text}",
        "응답형식 : ",
        '{"label":"DEF|RIGHT|PROC|ORG|CRIT|ETC", "reason":"분류 근거"}'        
    ]
    return "\n".join(base)

# 테스트 데이터: 실제 평가셋의 문장들 (Real Data)
test_sentences = [
    {
        'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
        'true_label': 'CRIT',
        'description': '자격/요건 규정'
    },
    {
        'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
        'true_label': 'ETC',
        'description': '시행일 규정'
    },
    {
        'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
        'true_label': 'DEF',
        'description': '정의 규정'
    }
]

results = []
print("hugging face 모델 one-shot / few-shot 비교")
print(f'모델 : {model_name}')
print(f'평가문장 : {len(test_sentences)}')
print(f'few-shot 예시 : {len(FEW_SHOT_EXAMPLES)}')

hugging face 모델 one-shot / few-shot 비교
모델 : Qwen/Qwen2.5-0.5B-Instruct
평가문장 : 3
few-shot 예시 : 3


In [47]:
test_sentences[0]

{'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
 'true_label': 'CRIT',
 'description': '자격/요건 규정'}

In [48]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [49]:
test_sentences[0]

{'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
 'true_label': 'CRIT',
 'description': '자격/요건 규정'}

In [53]:
def generate_hf_model(text, generate_kwargs=None,mode = 'one'):
    prompt = build_hf_prompt(text,mode,examples=FEW_SHOT_EXAMPLES if mode == 'few' else None)
    messages = [
        {"role":"system","content": SYSTEM_PROMPT },
        {"role":"user", "content" : prompt}
    ]
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    if not generate_kwargs :
        generate_kwargs = {
            'max_new_tokens':512,
            'temperature':0.3,
            'top_p':0.9
        }
    generated_ids = model.generate(
        **model_inputs,   
        **generate_kwargs             
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [51]:
test_sentences[0]

{'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
 'true_label': 'CRIT',
 'description': '자격/요건 규정'}

In [54]:
# 파라메터 튜닝
# temperature, top_p ,max_output_tokens
HF_EXPERIMENTS = [
    {
        'name':'deterministic',
        'do_sample' : False,
        'temperature' : 0.0,
        'top_p' : 1.0,
        'max_new_tokens':120,
        'description' : '가장 안정적인 설정'
    },
    {
        'name':'blanced',
        'do_sample' : True,
        'temperature' : 0.3,
        'top_p' : 0.9,
        'max_new_tokens':120,
        'description' : '균형잡힌 설정'
    },
    {
        'name':'creative',
        'do_sample' : True,
        'temperature' : 0.8,
        'top_p' : 0.95,
        'max_new_tokens':120,
        'description' : '변동성을 일부 허용하는 설정'
    }
]

TUNING_SAMPLES = [
    {
        'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
        'true_label': 'CRIT',
        'note': '수치 기준이 있는 요건 조항'
    },
    {
        'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
        'true_label': 'ETC',
        'note': '시행일을 정하는 조항'
    },
    {
        'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
        'true_label': 'DEF',
        'note': '개념의 뜻을 정의하는 조항'
    }
]

results = []
for exp in HF_EXPERIMENTS:
    sample = TUNING_SAMPLES[0]
    text = sample['text']
    kwargs = {
        'do_sample':exp['do_sample'],
        'temperature':exp['temperature'],
        'top_p':exp['top_p'],
        'max_new_tokens':exp['max_new_tokens']
    }
    result = generate_hf_model(text,kwargs,mode='few')
    results.append(result)

In [56]:
results, TUNING_SAMPLES[0]

(['{"label":"DEF", "reason":"분류 근거"}',
  '{"label":"DEF", "reason":"분류 근거"}',
  '{"label": "DEF", "reason": "정의/목적/적용범위 조항"}'],
 {'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
  'true_label': 'CRIT',
  'note': '수치 기준이 있는 요건 조항'})

### Few Shot 개수를늘려서 성능 비교(개수에 따른 성능비교)

In [58]:
def extract_json_object(text):
    """모델 응답에서 JSON 객체만 안전하게 추출한다."""
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end <= start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None

FEW_SHOT_VARIANTS = {
    0: [],
    1: [FEW_SHOT_EXAMPLES[0]],
    3: FEW_SHOT_EXAMPLES[:3],
    5: FEW_SHOT_EXAMPLES + [
        {
            'text': '이 법 시행에 필요한 사항은 대통령령으로 정한다.',
            'output': {'label': 'ETC', 'reason': '시행에 필요한 사항을 위임하는 조항'}
        },
        {
            'text': '모든 국민은 법 앞에 평등하며 성별, 종교 또는 사회적 신분에 의하여 차별을 받지 아니한다.',
            'output': {'label': 'RIGHT', 'reason': '평등권과 차별 금지를 규정하는 조항'}
        }
    ]
}

EVAL_SAMPLES = [
    {
        'text': '허가를 받으려는 자는 자본금 1억원 이상과 전담 인력 2명 이상을 갖추어야 한다.',
        'true_label': 'CRIT',
        'note': '수치 기준이 있는 요건 조항'
    },
    {
        'text': '이 법은 공포 후 6개월이 경과한 날부터 시행한다.',
        'true_label': 'ETC',
        'note': '시행일을 정하는 조항'
    },
    {
        'text': '공공기관이란 국가기관, 지방자치단체 및 법령에 따라 설치된 기관을 말한다.',
        'true_label': 'DEF',
        'note': '개념의 뜻을 정의하는 조항'
    },
    {
        'text': '누구든지 정당한 사유 없이 타인의 통신비밀을 침해하여서는 아니 된다.',
        'true_label': 'RIGHT',
        'note': '금지와 보호의무를 함께 가진 조항'
    },
    {
        'text': '장관은 위반 사실을 조사한 후 청문 절차를 거쳐 등록을 취소할 수 있다.',
        'true_label': 'PROC',
        'note': '조사와 청문 절차가 포함된 조항'
    }
]


def build_hf_fewshot_prompt(text, examples):
    """예시 목록을 받아 Few-Shot 프롬프트를 만든다.

    반환값:
        str: 예시 개수가 반영된 최종 프롬프트
    """
    return build_hf_prompt(text, mode='few' if examples else 'zero', examples=examples)


def predict_with_hf(text, examples):
    """Hugging Face 모델로 단일 문장을 분류한다.

    반환값:
        dict: label, reason, raw_response를 포함한 결과
    """
    prompt = build_hf_fewshot_prompt(text, examples)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt}
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([chat_text], return_tensors='pt').to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )
    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    parsed = extract_json_object(raw_response)
    return {
        'raw_response': raw_response,
        'label': parsed.get('label') if parsed else 'PARSE_FAIL',
        'reason': parsed.get('reason') if parsed else raw_response[:160]
    }


print('=' * 90)
print('Few-Shot 예시 개수별 성능 비교')
print('=' * 90)
print('고정 조건: 모델, 평가 문장, 출력 형식, generation 설정')
print('변화 조건: 예시 개수(0, 1, 3, 5)')

fewshot_results = []

for example_count, examples in FEW_SHOT_VARIANTS.items():
    print('\n' + '-' * 90)
    print(f'예시 개수: {example_count}개')
    print('-' * 90)
    
    correct_count = 0
    for sample in EVAL_SAMPLES:
        result = predict_with_hf(sample['text'], examples)
        predicted_label = result['label']
        is_correct = predicted_label == sample['true_label']
        correct_count += int(is_correct)
        
        fewshot_results.append({
            'example_count': example_count,
            'text': sample['text'],
            'true_label': sample['true_label'],
            'predicted_label': predicted_label,
            'correct': is_correct,
            'raw_response': result['raw_response']
        })
        
        print(f"문장: {sample['note']}")
        print(f"정답: {sample['true_label']} / 예측: {predicted_label} / 정답 여부: {'O' if is_correct else 'X'}")
        print(f"이유: {result['reason']}")
        print('-' * 40)
    
    accuracy = correct_count / len(EVAL_SAMPLES)
    print(f'예시 {example_count}개 정확도: {accuracy:.2f}')

fewshot_df = pd.DataFrame(fewshot_results)
fewshot_summary = (
    fewshot_df
    .groupby('example_count', as_index=False)['correct']
    .mean()
    .rename(columns={'correct': 'accuracy'})
    .sort_values('example_count')
)

print('\n' + '=' * 90)
print('예시 개수별 요약')
print('=' * 90)
fewshot_summary

Few-Shot 예시 개수별 성능 비교
고정 조건: 모델, 평가 문장, 출력 형식, generation 설정
변화 조건: 예시 개수(0, 1, 3, 5)

------------------------------------------------------------------------------------------
예시 개수: 0개
------------------------------------------------------------------------------------------
문장: 수치 기준이 있는 요건 조항
정답: CRIT / 예측: DEF / 정답 여부: X
이유: 분류 근거
----------------------------------------
문장: 시행일을 정하는 조항
정답: ETC / 예측: DEF / 정답 여부: X
이유: 분류 근거
----------------------------------------
문장: 개념의 뜻을 정의하는 조항
정답: DEF / 예측: ORG|PROC|DEF / 정답 여부: X
이유: 국가기관과 지방자치단체를 포함하고 있어, 설치된 기관에 대한 정의와 관계에 따른 규정이 포함되어 있습니다.
----------------------------------------
문장: 금지와 보호의무를 함께 가진 조항
정답: RIGHT / 예측: DEF / 정답 여부: X
이유: 문장의 목적 및 적용 범위에 따라 분류
----------------------------------------
문장: 조사와 청문 절차가 포함된 조항
정답: PROC / 예측: DEF|RIGHT|PROC|ORG|CRIT|ETC / 정답 여부: X
이유: 문장 내용은 권리를 지니고, 청문 절차와 등록 취소에 대한 의무가 있습니다.
----------------------------------------
예시 0개 정확도: 0.00

------------------------------------------------------------

,example_count,accuracy
0,0,0.0
1,1,0.2
2,3,0.0
3,5,0.2


### 성능올리기
- 예시를 더 추가하는 것보다 라벨 경계 규칙을 프롬프트에 명시하면 성능이 어떻게 달라지는지 비교
- 분류문제의 성능향상 샘플수 늘리는것도 중요하지만 기준을 선명하게 만드는 것이 주요
- 오답분석 관점에서 규칙강화의 효과

- 포인트
    - 기본 프롬프트와 규칙강화 프롬프트의 차이?
    - PROC, ETC, RIGHT, CRIT처럼 경계가 애매한 라벨에서 변화?
    - 출력 형식 고정이되면 안정성에 영향?